In [1]:
import os
import pandas as pd
import numpy as np

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)

df_query = pd.read_csv("merged_cancer_targets_260601.csv")

df_query.head()

,gene_name,uniprot_id,recommendedName,Pacini_in_list,oncokb_therapeutic,consmic_tier,fda_targets,pharma
0,IGH,A2N192,NaN,NaN,NaN,1.0,NaN,NaN
1,PSMB11,A5LHX3,Proteasome subunit beta type-11,True,NaN,NaN,NaN,NaN
2,HTR3E,A5X5Y0,5-hydroxytryptamine receptor 3E,True,NaN,NaN,True,NaN
3,NUTM2B,A6NNL0,NUT family member 2B,NaN,NaN,1.0,NaN,NaN
4,TUBB8B,A6NNZ2,Tubulin beta 8B,True,NaN,NaN,NaN,NaN


In [2]:
# Reformat df_query with columns: label, protein_name, uniprot_id
df_query = df_query[["pharma", "gene_name", "uniprot_id"]]

# rename columns
df_query.columns = ["label", "protein_name", "uniprot_id"]

df_query.head()


,label,protein_name,uniprot_id
0,NaN,IGH,A2N192
1,NaN,PSMB11,A5LHX3
2,NaN,HTR3E,A5X5Y0
3,NaN,NUTM2B,A6NNL0
4,NaN,TUBB8B,A6NNZ2


In [3]:
import requests

row = df_query.iloc[0]
uniprot_id = row["uniprot_id"]

def get_pdb_accessions(uniprot_id):
    """Return unique PDB IDs cross-referenced for a UniProt accession."""
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
    except (requests.RequestException, ValueError) as exc:
        print(f"  {uniprot_id}: UniProt fetch failed ({exc})")
        return []

    pdb_ids = []
    seen = set()
    for dbref in data.get("uniProtKBCrossReferences", []):
        if dbref.get("database") != "PDB":
            continue
        pdb_id = dbref.get("id")
        if pdb_id and pdb_id not in seen:
            seen.add(pdb_id)
            pdb_ids.append(pdb_id.upper())
    return pdb_ids


get_pdb_accessions(uniprot_id)

[]

In [4]:
RCSB_CORE_API = "https://data.rcsb.org/rest/v1/core"


def _get_chemcomp_smiles(comp_id):
    """Fetch SMILES for a chemical component ID."""
    try:
        response = requests.get(f"{RCSB_CORE_API}/chemcomp/{comp_id}", timeout=30)
        response.raise_for_status()
        descriptor = response.json().get("rcsb_chem_comp_descriptor") or {}
        return descriptor.get("SMILES_stereo") or descriptor.get("SMILES")
    except requests.RequestException:
        return None


def _build_polymer_uniprot_maps(pdb_id, polymer_entity_ids):
    """
    Map polymer entity_id and asym_id -> UniProt accessions for a PDB entry.
    """
    entity_to_uniprot = {}
    asym_to_uniprot = {}

    for entity_id in polymer_entity_ids:
        try:
            response = requests.get(
                f"{RCSB_CORE_API}/polymer_entity/{pdb_id}/{entity_id}",
                timeout=30,
            )
            if response.status_code != 200:
                continue
            container = response.json().get("rcsb_polymer_entity_container_identifiers") or {}
        except (requests.RequestException, ValueError, TypeError, KeyError):
            continue
        uniprot_ids = container.get("uniprot_ids") or []
        entity_to_uniprot[str(entity_id)] = uniprot_ids

        for asym_id in (container.get("asym_ids") or []) + (container.get("auth_asym_ids") or []):
            asym_to_uniprot[asym_id] = uniprot_ids

    return entity_to_uniprot, asym_to_uniprot


def _neighbor_uniprot_ids(neighbor, entity_to_uniprot, asym_to_uniprot):
    """Resolve a target neighbor record to UniProt accessions."""
    uniprot_ids = set()

    target_entity_id = neighbor.get("target_entity_id")
    if target_entity_id is not None:
        for uid in entity_to_uniprot.get(str(target_entity_id), []):
            uniprot_ids.add(uid)

    target_asym_id = neighbor.get("target_asym_id")
    if target_asym_id is not None:
        for uid in asym_to_uniprot.get(target_asym_id, []):
            uniprot_ids.add(uid)

    return uniprot_ids


def _is_subject_of_investigation(entity_data):
    """True if the non-polymer entity is annotated as SUBJECT_OF_INVESTIGATION."""
    for annotation in entity_data.get("rcsb_nonpolymer_entity_annotation") or []:
        if annotation.get("type") == "SUBJECT_OF_INVESTIGATION":
            return True

    for feature in entity_data.get("rcsb_nonpolymer_entity_feature") or []:
        if feature.get("type") == "SUBJECT_OF_INVESTIGATION":
            return True

    return False


def get_ligands_from_pdb(pdb_id, uniprot_id, filter_by_subject_of_investigation = False):
    """
    Query RCSB PDB API for ligands in a structure that contact the query protein.

    Workflow:
      1. entry/{pdb_id} -> non_polymer_entity_ids
      2. nonpolymer_entity/{pdb_id}/{entity_id} -> SOI annotation + instance asym_ids
      3. nonpolymer_entity_instance/{pdb_id}/{asym_id} -> rcsb_target_neighbors
      4. Map neighbor target_entity_id / target_asym_id -> polymer UniProt IDs

    A ligand is kept only if it is SUBJECT_OF_INVESTIGATION (SOI) and has at least
    one target neighbor on a polymer chain mapped to the query UniProt accession.

    Parameters:
        pdb_id (str): PDB entry ID (e.g. "8VLB").
        uniprot_id (str): Query UniProt accession to filter binding context.

    Returns:
        list of dict with keys: ligand_code, ligand_name, smiles
    """
    pdb_id = pdb_id.upper()
    uniprot_id = uniprot_id.upper()

    try:
        entry_response = requests.get(f"{RCSB_CORE_API}/entry/{pdb_id}", timeout=30)
        if entry_response.status_code != 200:
            return []
        container = entry_response.json().get("rcsb_entry_container_identifiers") or {}
    except (requests.RequestException, ValueError, TypeError, KeyError):
        return []

    nonpolymer_entity_ids = container.get("non_polymer_entity_ids") or []
    polymer_entity_ids = container.get("polymer_entity_ids") or []
    if not nonpolymer_entity_ids:
        return []

    entity_to_uniprot, asym_to_uniprot = _build_polymer_uniprot_maps(
        pdb_id, polymer_entity_ids
    )

    ligand_rows = []
    seen_comp_ids = set()

    for entity_id in nonpolymer_entity_ids:
        try:
            entity_response = requests.get(
                f"{RCSB_CORE_API}/nonpolymer_entity/{pdb_id}/{entity_id}",
                timeout=30,
            )
            if entity_response.status_code != 200:
                continue
            entity_data = entity_response.json()
        except (requests.RequestException, ValueError, TypeError, KeyError):
            continue
        comp_id = (entity_data.get("pdbx_entity_nonpoly") or {}).get("comp_id")
        comp_name = (entity_data.get("pdbx_entity_nonpoly") or {}).get("name")
        if not comp_id or comp_id in seen_comp_ids:
            continue
        
        if filter_by_subject_of_investigation and not _is_subject_of_investigation(entity_data):
            continue

        instance_asym_ids = (
            (entity_data.get("rcsb_nonpolymer_entity_container_identifiers") or {}).get(
                "asym_ids"
            )
            or []
        )

        binds_query_protein = False
        for asym_id in instance_asym_ids:
            try:
                instance_response = requests.get(
                    f"{RCSB_CORE_API}/nonpolymer_entity_instance/{pdb_id}/{asym_id}",
                    timeout=30,
                )
                if instance_response.status_code != 200:
                    continue
                neighbors = instance_response.json().get("rcsb_target_neighbors") or []
            except (requests.RequestException, ValueError, TypeError, KeyError):
                continue
            for neighbor in neighbors:
                neighbor_uniprots = _neighbor_uniprot_ids(
                    neighbor, entity_to_uniprot, asym_to_uniprot
                )
                if uniprot_id in neighbor_uniprots:
                    binds_query_protein = True
                    break
            if binds_query_protein:
                break

        if not binds_query_protein:
            continue

        seen_comp_ids.add(comp_id)
        ligand_rows.append(
            {
                "ligand_code": comp_id,
                "ligand_name": comp_name,
                "smiles": _get_chemcomp_smiles(comp_id),
            }
        )

    return ligand_rows


# Example: 3JF is SOI and binds VHL (P40337) and CDO1 (Q16878) on 8VLB
get_ligands_from_pdb("8VLB", "P40337")

[{'ligand_code': '3JF',
  'ligand_name': 'N-acetyl-3-methyl-L-valyl-(4R)-4-hydroxy-N-[4-(4-methyl-1,3-thiazol-5-yl)benzyl]-L-prolinamide',
  'smiles': 'Cc1c(scn1)c2ccc(cc2)CNC(=O)[C@@H]3C[C@H](CN3C(=O)[C@H](C(C)(C)C)NC(=O)C)O'},
 {'ligand_code': 'CIT',
  'ligand_name': 'CITRIC ACID',
  'smiles': 'C(C(=O)O)C(CC(=O)O)(C(=O)O)O'}]

In [5]:
def build_ligand_df_for_protein(row_in):
    """
    Build a DataFrame listing all PDBs and ligands for a wishlist protein.

    Args:
        row_in (pd.Series): A row from df_wishlist.

    Returns:
        pd.DataFrame: DataFrame with columns [label, protein_name, uniprot_id, pdb_id, ligand_code, ligand_name, smiles].
    """
    label = row_in["label"]
    protein_name = row_in["protein_name"]
    uniprot_id = row_in["uniprot_id"]
    pdb_ids = get_pdb_accessions(uniprot_id)
    if not pdb_ids:
        return pd.DataFrame()

    df_out = pd.DataFrame()
    for pdb_id in pdb_ids:
        try:
            ligands = get_ligands_from_pdb(pdb_id, uniprot_id)
        except (requests.RequestException, TypeError, KeyError, AttributeError) as exc:
            print(f"  {uniprot_id} {pdb_id}: skipped ({exc})")
            continue

        if len(ligands) == 0:
            row_out = {
                "label": label,
                "protein_name": protein_name,
                "uniprot_id": uniprot_id,
                "pdb_id": pdb_id,
                "ligand_code": None,
                "ligand_name": None,
                "smiles": None, 
            }
            df_out = pd.concat([df_out, pd.DataFrame([row_out])], ignore_index=True)
        else:
            for ligand in ligands:
                row_out = {
                    "label": label,
                    "protein_name": protein_name,
                    "uniprot_id": uniprot_id,
                    "pdb_id": pdb_id,
                    "ligand_code": ligand["ligand_code"],
                    "ligand_name": ligand["ligand_name"],
                    "smiles": ligand["smiles"], 
                }
                df_out = pd.concat([df_out, pd.DataFrame([row_out])], ignore_index=True)

    return df_out

# Example usage for first wishlist protein:
# df_out = build_ligand_df_for_protein(df_wishlist.iloc[0])


In [14]:
from pathlib import Path
import subprocess

# Hardcoded defaults for reruns and downstream notebook cells
input_txt = Path("data/query_uniprot_ids.txt")
output_csv = Path("data/all_ligands.csv")
blacklist_txt = Path("blacklist.txt")
slurm_script = Path("scripts/slurm/uniprot_to_pdb_ligand.sh")

# Write one UniProt ID per line for the batch script input
uniprot_ids = (
    df_query["uniprot_id"]
    .dropna()
    .astype(str)
    .str.strip()
)
uniprot_ids = uniprot_ids[uniprot_ids != ""]
uniprot_ids = uniprot_ids.drop_duplicates(keep="first")
input_txt.write_text("\n".join(uniprot_ids) + "\n", encoding="utf-8")

submit_cmd = [
    "sbatch",
    str(slurm_script),
    str(input_txt),
    str(output_csv),
    str(blacklist_txt),
]
result = subprocess.run(submit_cmd, check=True, capture_output=True, text=True)
print(result.stdout.strip())
print(f"Prepared {len(uniprot_ids)} unique UniProt IDs in {input_txt}")
print("Monitor logs: logs/uniprot_ligands-<jobid>.out")


Submitted batch job 54875004
Prepared 2026 unique UniProt IDs in data/query_uniprot_ids.txt
Monitor logs: logs/uniprot_ligands-<jobid>.out


In [35]:
import pandas as pd
from pathlib import Path

output_csv = Path("all_ligands.csv")
if not output_csv.exists():
    raise FileNotFoundError(
        f"{output_csv} not found yet. Wait for SLURM job completion, then rerun this cell."
    )

df_all = pd.read_csv(output_csv)
print(f"Loaded {len(df_all)} rows from {output_csv}")
df_all.head()

,client,protein_name,uniprot_id,pdb_id,ligand_code,ligand_name,smiles,formula,mw
0,AZ,NRF2,Q16236,2FLU,None,None,None,None,NaN
1,AZ,NRF2,Q16236,2LZ1,None,None,None,None,NaN
2,AZ,NRF2,Q16236,3ZGC,None,None,None,None,NaN
3,AZ,NRF2,Q16236,4IFL,None,None,None,None,NaN
4,AZ,NRF2,Q16236,5WFV,None,None,None,None,NaN
...,...,...,...,...,...,...,...,...,...
1733,Merck,TAF15,Q16514,8WAO,None,None,None,None,NaN
1734,Merck,TAF15,Q16514,8WAP,None,None,None,None,NaN
1735,Merck,TAF15,Q16514,8WAQ,None,None,None,None,NaN
1736,Merck,TAF15,Q16514,8WAR,None,None,None,None,NaN


In [36]:
# Function to convert smiles to molecular formula and molecular weight using RDKit
from rdkit import Chem

def smiles_to_formula(smiles):
    """Convert SMILES to molecular formula using RDKit's rdMolDescriptors."""
    from rdkit.Chem import rdMolDescriptors
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return rdMolDescriptors.CalcMolFormula(mol)

def smiles_to_mw(smiles):
    """Convert SMILES to molecular weight using RDKit's rdMolDescriptors."""
    from rdkit.Chem import rdMolDescriptors
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return rdMolDescriptors.CalcExactMolWt(mol)

smiles = "C1=CC=CC=C1"
print("Formula:", smiles_to_formula(smiles))
print("Exact MW:", smiles_to_mw(smiles))


Formula: C6H6
Exact MW: 78.046950192


In [37]:
# Add formula and molecular weight to df_all, return None if smiles is None
df_all["formula"] = df_all["smiles"].apply(lambda s: smiles_to_formula(s) if s is not None else None)
df_all["mw"] = df_all["smiles"].apply(lambda s: smiles_to_mw(s) if s is not None else None)

df_all.to_csv("all_ligands.csv", index=False)
df_all


,client,protein_name,uniprot_id,pdb_id,ligand_code,ligand_name,smiles,formula,mw
0,AZ,NRF2,Q16236,2FLU,None,None,None,None,NaN
1,AZ,NRF2,Q16236,2LZ1,None,None,None,None,NaN
2,AZ,NRF2,Q16236,3ZGC,None,None,None,None,NaN
3,AZ,NRF2,Q16236,4IFL,None,None,None,None,NaN
4,AZ,NRF2,Q16236,5WFV,None,None,None,None,NaN
...,...,...,...,...,...,...,...,...,...
1733,Merck,TAF15,Q16514,8WAO,None,None,None,None,NaN
1734,Merck,TAF15,Q16514,8WAP,None,None,None,None,NaN
1735,Merck,TAF15,Q16514,8WAQ,None,None,None,None,NaN
1736,Merck,TAF15,Q16514,8WAR,None,None,None,None,NaN


In [38]:
# For each unique ligand_code in df_all, check how many uniprot_id are associated with it
df_ligand_uniprot_count = df_all.groupby("ligand_code")["uniprot_id"].nunique().reset_index(name="uniprot_id_count")

# Filter ligands to those that are associated with only one uniprot_id - these are  ligands specific to one protein
specific_ligands = df_ligand_uniprot_count[df_ligand_uniprot_count["uniprot_id_count"] == 1]["ligand_code"].tolist()
specific_ligands



['056',
 '12I',
 '17H',
 '17W',
 '198',
 '1KI',
 '2MI',
 '30Z',
 '3B6',
 '3E0',
 '3LS',
 '3OD',
 '4HY',
 '4MQ',
 '51Y',
 '5FW',
 '5UD',
 '6B3',
 '6SM',
 '77T',
 '77U',
 '7TT',
 '92O',
 '946',
 '97A',
 '9FG',
 '9GH',
 '9GK',
 '9GN',
 '9GQ',
 '9GT',
 '9GW',
 '9GZ',
 '9H2',
 '9H5',
 '9JK',
 '9JT',
 'A1A1Q',
 'A1ACS',
 'A1AM2',
 'A1ANM',
 'A1ANN',
 'A1API',
 'A1APJ',
 'A1APK',
 'A1APS',
 'A1APT',
 'A1ARO',
 'A1ARZ',
 'A1ATT',
 'A1B9H',
 'A1BAF',
 'A1BLD',
 'A1CI0',
 'A1D5Y',
 'A1D5Z',
 'A1D9F',
 'A1EMT',
 'A1I41',
 'A1IH3',
 'A1L13',
 'A1LYN',
 'ACE',
 'ACP',
 'ALE',
 'ANP',
 'APR',
 'ARS',
 'AV6',
 'B1I',
 'B5R',
 'B66',
 'B67',
 'B68',
 'BHM',
 'CA4',
 'CGO',
 'CIT',
 'CMC',
 'CNA',
 'CO',
 'CO3',
 'COA',
 'CPT',
 'CU',
 'CU1',
 'CYS',
 'D7Z',
 'DHT',
 'DSN',
 'ENM',
 'EXN',
 'EXQ',
 'EY2',
 'EYB',
 'EYE',
 'FHM',
 'FLF',
 'FOR',
 'FUC',
 'FY8',
 'GLC',
 'GOH',
 'HFT',
 'HR5',
 'I2G',
 'ICO',
 'IZ8',
 'IZM',
 'IZV',
 'J8V',
 'J8Y',
 'J91',
 'J97',
 'JAD',
 'JKI',
 'K0J',
 'K10',
 'K3J',


In [39]:
# Filter df_all to only include these ligands
df_specific_ligands = df_all[df_all["ligand_code"].isin(specific_ligands)]

# Extract the number following 'C' in the formula (e.g., C12H10O2 would yield 12 for carbon count).
import re
def extract_carbon_count(formula):
    if not formula:
        return 0
    match = re.search(r'C(\d*)', formula)
    if not match:
        return 0
    count = match.group(1)
    return int(count) if count else 1  # If just 'C', assume 1
df_specific_ligands["num_carbon"] = df_specific_ligands["formula"].apply(extract_carbon_count)

# Remove ligands with no carbon atoms
df_specific_ligands = df_specific_ligands[df_specific_ligands["num_carbon"] > 0]

# Remove ligands with mw < 120
df_specific_ligands = df_specific_ligands[df_specific_ligands["mw"] > 120]

print(f"df_specific_ligands.shape: {df_specific_ligands.shape}")
df_specific_ligands



df_specific_ligands.shape: (340, 10)


/tmp/54597086/ipykernel_2920657/215907688.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_specific_ligands["num_carbon"] = df_specific_ligands["formula"].apply(extract_carbon_count)


,client,protein_name,uniprot_id,pdb_id,ligand_code,ligand_name,smiles,formula,mw,num_carbon
15,AZ,NRF2,Q16236,7X5E,P6G,HEXAETHYLENE GLYCOL,C(COCCOCCOCCOCCOCCO)O,C12H26O7,282.167853,12
40,AZ,DNp63,Q9H3D4,4A9Z,PE4,2-{2-[2-(2-{2-[2-(2-ETHOXY-ETHOXY)-ETHOXY]-ETH...,CCOCCOCCOCCOCCOCCOCCOCCO,C16H34O8,354.225368,16
42,AZ,DNp63,Q9H3D4,6RU6,ACP,PHOSPHOMETHYLPHOSPHONIC ACID ADENYLATE ESTER,c1nc(c2c(n1)n(cn2)[C@H]3[C@@H]([C@@H]([C@H](O3...,C11H18N5O12P3,505.016481,11
91,AZ,CTNNB1,P35222,6M90,J91,2-(2-fluorophenoxy)-3-{[2-oxo-6-(trifluorometh...,c1ccc(c(c1)Oc2c(cccc2NC(=O)C3=CC=C(NC3=O)C(F)(...,C20H12F4N2O5,436.068234,20
92,AZ,CTNNB1,P35222,6M91,J97,"3-({4-[(2,6-dichlorophenyl)sulfanyl]-2-oxo-6-(...",c1cc(cc(c1)NC(=O)C2=C(C=C(NC2=O)C(F)(F)F)Sc3c(...,C20H11Cl2F3N2O4S,501.976868,20
...,...,...,...,...,...,...,...,...,...,...
1655,Merck,aSyn,P37840,8XWD,KDH,"(2R,3R)-5,7-dihydroxy-2-(3,4,5-trihydroxypheny...",c1c(cc(c(c1O)O)O)[C@@H]2[C@@H](Cc3c(cc(cc3O2)O...,C22H18O11,458.084911,22
1658,Merck,aSyn,P37840,8ZLI,A1L13,"~{N},~{N}-dimethyl-4-(6-methyl-1,3-benzothiazo...",Cc1ccc2c(c1)sc(n2)c3ccc(cc3)N(C)C,C16H16N2S,268.103420,16
1659,Merck,aSyn,P37840,8ZLO,1KI,2-bromanyl-4-[(~{E})-2-[6-[2-(2-fluoranylethox...,Cc1cc2c(cc1N(C)CCOCCF)sc(n2)/C=C/c3ccc(c(c3)Br)O,C21H22BrFN2O2S,464.056939,21
1661,Merck,aSyn,P37840,8ZMY,1KI,2-bromanyl-4-[(~{E})-2-[6-[2-(2-fluoranylethox...,Cc1cc2c(cc1N(C)CCOCCF)sc(n2)/C=C/c3ccc(c(c3)Br)O,C21H22BrFN2O2S,464.056939,21


In [40]:
df_specific_ligands.to_csv("specific_ligands.csv", index=False)